# 02 Dataset Cleaning and Preprocessing

This notebook profiles, assesses, cleans, validates, and exports the merged webcam-based PM2.5 dataset.

Input: `data/interim/merged_dataset.csv`

Output: `data/processed/final_dataset.csv`

## Workflow Overview
1. Data loading  
2. Data profiling  
3. Data quality assessment  
4. Data cleaning  
5. Validation and visualization  
6. Export and summary

## Setup

In [112]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")
sns.set_theme(style="whitegrid")

## 1. Data Loading

Load the merged dataset generated from the previous dataset construction pipeline.


In [ ]:
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "merged_dataset.csv"
)

# Load merged dataset
df_raw = pd.read_csv(INPUT_FILE)

print(f"Rows: {len(df_raw)}")
print(f"Columns: {df_raw.shape[1]}")


In [114]:
# Create working copy
df = df_raw.copy()

## 2. Data Profiling

Inspect dataset structure, temporal coverage, data types, and basic numerical summaries.

### 2.1 Basic structure

In [ ]:
df.info()
df.head()

### 2.2 Missing-value overview

Analyze missing values across all variables to identify incomplete observations and potential data quality issues.

In [ ]:
missing_summary = (
    pd.DataFrame({
        "missing_count": df_raw.isna().sum(),
        "missing_percent": df_raw.isna().mean() * 100,
    })
    .sort_values("missing_percent", ascending=False)
)

missing_summary.head(15)

In [ ]:
import missingno as mno

plt.figure(figsize=(10, 5))

mno.matrix(df)

plt.title("Missing Values Matrix")

plt.show()

### 2.3 Temporal coverage
Inspect temporal consistency, timestamp uniqueness, and continuity of the hourly observations.

In [ ]:
# Time range
df["time"] = pd.to_datetime(df["time"])

print("Start time:", df["time"].min())
print("End time:", df["time"].max())

print(
    "Unique timestamps:",
    df["time"].nunique()
)

In [ ]:
#Duplicate timestamps
duplicate_count = df["time"].duplicated().sum()

print("Duplicated timestamps:", duplicate_count)


In [ ]:
#Hourly continuity
expected_range = pd.date_range(
    start=df["time"].min(),
    end=df["time"].max(),
    freq="h"
)

missing_times = expected_range.difference(df["time"])

print("Expected timestamps:", len(expected_range))
print("Observed timestamps:", len(df["time"]))
print("Missing timestamps:", len(missing_times))

### 2.4 Numerical profiling

Explore numerical variables through descriptive statistics, distributions, and correlation analysis.

In [ ]:
# Descriptive statistics
numeric_summary = (
    df.select_dtypes(include="number")
    .describe()
    .T
)

numeric_summary

In [ ]:
#  Numerical distributions-image features
image_vars = [
    "R_roi",
    "G_roi",
    "B_roi",
    "S_mean",
    "B_R_ratio",
    "contrast",
]

df[image_vars].hist(
    figsize=(12, 8),
    bins=30
)

plt.tight_layout()

plt.show()

In [ ]:
# Numerical distributions-target + weather variables
weather_vars = [
    "PM25",
    "T2M",
    "RH",
    "WS10",
    "BLH",
]

df[weather_vars].hist(
    figsize=(10, 6),
    bins=30
)

plt.tight_layout()

plt.show()

In [ ]:
# Correlation analysis
corr_matrix = (
    df.select_dtypes(include="number")
    .corr()
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix")

plt.show()

## 3. Data Quality Assessment

Assess missing values, invalid physical values, duplicates, and potential outliers before applying cleaning rules.

### 3.1 Missing-value assessment

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0]

missing_summary.sort_values(by="missing_percent", ascending=False)

### 3.2 Duplicate assessment


In [ ]:
print("Duplicated rows:", df_raw.duplicated().sum())
print("Duplicated timestamps:", df_raw["time"].duplicated().sum())

### 3.3 Invalid-value assessment

In [ ]:
checks = {}

# PM2.5
checks["PM25_negative"] = (df["PM25"] < 0).sum()

# Relative humidity
checks["ERA5_RH_outside_0_100"] = (
    (df["RH"] < 0) |(df["RH"] > 100)).sum()

checks["ARPA_RH_outside_0_100"] = (
    (df["relative_humidity_mean"] < 0) |(df["relative_humidity_mean"] > 100)).sum()

# Total cloud cover
checks["TCC_outside_0_1"] = (
    (df["TCC"] < 0) |(df["TCC"] > 1)).sum()

# Precipitation
checks["TP_negative"] = (df["TP"] < 0).sum()

# Wind speed
checks["wind_speed_mean_negative"] = (df["wind_speed_mean"] < 0).sum()

checks["WS10_negative"] = (df["WS10"] < 0).sum()

# Convert to dataframe
invalid_summary = pd.DataFrame.from_dict(
    checks,orient="index",columns=["count"])

invalid_summary

### 3.4 Outlier assessment

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

outlier_rows = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = int(
        ((df[col] < lower) | (df[col] > upper)).sum()
    )

    outlier_rows.append({
        "column": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "iqr_outlier_count": count,
    })

outlier_summary = pd.DataFrame(outlier_rows).sort_values(
    "iqr_outlier_count",
    ascending=False
)

outlier_summary.head(15)

In [ ]:
selected_vars = [
    "PM25",
    "B_R_ratio",
    "contrast",
    "T2M",
    "RH",
    "WS10",
]

plt.figure(figsize=(12, 6))

df[selected_vars].boxplot()

plt.xticks(rotation=45)

plt.show()

### 3.5 Temporal consistency assessment

In [130]:
expected_range = pd.date_range(
    start=df["time"].min(),
    end=df["time"].max(),
    freq="h"
)

missing_times = expected_range.difference(df["time"])

print(
    "Missing timestamps:",
    len(missing_times)
)

Missing timestamps: 64


## 4. Data Cleaning

Apply data type conversion, missing-value handling, invalid-value correction, deduplication, and feature transformation.

### 4.1 Data Transformation and Standardization

Standardize the dataset structure by converting time fields and sorting observations chronologically.

In [131]:
df["time"] = pd.to_datetime(df["time"],errors="coerce")

print( "Invalid time values:",df["time"].isna().sum())

df = df.sort_values("time").reset_index(drop=True)

print("Time range:",df["time"].min(),"to",df["time"].max())

print( "Rows:",len(df))

Invalid time values: 0
Time range: 2026-03-01 03:00:00 to 2026-03-12 22:00:00
Rows: 220


### 4.2 Missing-Value Handling

Handle missing values using variable-specific interpolation and filling strategies.

In [ ]:
missing_before = df.isna().sum().sum()

# Linear interpolation for continuous hourly variables
linear_interpolation_columns = [
    "wind_speed_mean",
    "CBH",
]

for col in linear_interpolation_columns:
    if col in df.columns:
        df[col] = df[col].interpolate(
            method="linear",
            limit_direction="both"
        )

# Wind direction is circular data
if "wind_direction_mean" in df.columns:
    df["wind_direction_mean"] = (
        df["wind_direction_mean"]
        .ffill()
        .bfill()
    )

missing_after = df.isna().sum().sum()

print("Missing values before:", missing_before)
print("Missing values after:", missing_after)

Missing values before: 76
Missing values after: 0


In [133]:
missing_after_summary = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percent": (df.isna().mean() * 100).round(2).values
})

missing_after_summary = missing_after_summary[
    missing_after_summary["missing_count"] > 0
].sort_values(
    by="missing_percent",
    ascending=False
).reset_index(drop=True)

if missing_after_summary.empty:
    print("No missing values remain.")
else:
    display(missing_after_summary)

No missing values remain.


### 4.3 Invalid-Value Correction

Remove observations containing physically invalid or unrealistic values.

In [134]:
initial_rows = len(df)

valid_mask = (
    (df["PM25"] >= 0) &
    (df["RH"].between(0, 100)) &
    (df["relative_humidity_mean"].between(0, 100)) &
    (df["TCC"].between(0, 1)) &
    (df["TP"] >= 0) &
    (df["wind_speed_mean"] >= 0) &
    (df["WS10"] >= 0)
)

df = df[valid_mask].reset_index(drop=True)

print("Rows before invalid-value filtering:", initial_rows)
print("Rows after invalid-value filtering:", len(df))
print("Rows removed:", initial_rows - len(df))

Rows before invalid-value filtering: 220
Rows after invalid-value filtering: 220
Rows removed: 0


### 4.4 Remove Non-Modeling Columns

Remove columns that are not useful for numerical modeling.

In [135]:
columns_to_drop = [
    "image_path",
    "Unit",
]

df = df.drop(
    columns=columns_to_drop
)

print("Dropped columns:", columns_to_drop)

print("Shape after dropping columns:", df.shape)

Dropped columns: ['image_path', 'Unit']
Shape after dropping columns: (220, 32)


### 4.5 Duplicate Removal

Check and remove duplicated timestamp observations.

In [139]:
rows_before = len(df)

df = df.drop_duplicates(subset="time").reset_index(drop=True)

rows_after = len(df)

print("Duplicate timestamps removed:",rows_before - rows_after)

print("Rows after duplicate removal:", rows_after)

Duplicate timestamps removed: 0
Rows after duplicate removal: 220


### 4.6 Numeric Conversion

Convert numeric-like variables to proper numeric formats for analysis and modeling.

In [136]:
df = df.replace("<NA>", pd.NA)

for col in df.columns:
    if col != "time":
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

print("Numeric conversion completed.")
print("Remaining missing values:", df.isna().sum().sum())

Numeric conversion completed.
Remaining missing values: 0


### 4.7 Temporal and Wind-Direction Feature Engineering

Generate additional temporal and circular features to better represent periodic environmental patterns.

In [ ]:
# These features are useful because air pollution and illumination have daily cycles.
df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["weekday"] = df["time"].dt.weekday
df["month"] = df["time"].dt.month

# Cyclical encoding for hour.
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# A simple daylight flag for preliminary analysis.
# This is not a precise astronomical sunrise/sunset calculation, but is useful for the first model iteration.
df["is_daytime"] = ((df["hour"] >= 6) & (df["hour"] <= 18)).astype(int)

df[["time", "hour", "weekday", "hour_sin", "hour_cos", "is_daytime"]].head()

In [ ]:
# Convert wind direction to circular features

df["wind_dir_sin"] = np.sin(
    np.radians(df["wind_direction_mean"]))

df["wind_dir_cos"] = np.cos(
    np.radians(df["wind_direction_mean"]))

df[[ "wind_direction_mean","wind_dir_sin", "wind_dir_cos"]].head()

## 5. Validation & Visualization

Validate the cleaned dataset and visualize key variables after preprocessing.

### 5.1 Final dataset validation

In [ ]:
print("Final shape:", df.shape)
print("Final missing values:", df.isna().sum().sum())
print("Duplicated timestamps:", df["time"].duplicated().sum())
df.head()

### 5.2 Boxplots

Visualize feature distributions and potential outliers after cleaning.

In [ ]:
important_columns = [
    "PM25", "R_roi","G_roi","B_roi","B_R_ratio","contrast","RH","T2M","WS10",
    "BLH", "temperature_mean","relative_humidity_mean","wind_speed_mean",]

plot_columns = [
    col for col in important_columns
    if col in df.columns]

plt.figure(figsize=(14, 6))

df[plot_columns].boxplot(rot=90)

plt.title("Boxplot of Important Variables After Cleaning")

plt.tight_layout()

plt.show()

### 5.3 Correlation with PM2.5

Inspect the relationship between PM2.5 and selected environmental variables.

In [ ]:
corr_with_pm25 = (
    df
    .select_dtypes(include=np.number)
    .corr()["PM25"]
    .sort_values(ascending=False)
)

print("Top positive correlations with PM2.5:")
display(corr_with_pm25.head(10))

print("\nTop negative correlations with PM2.5:")
display(corr_with_pm25.tail(10))

### 5.4 PM2.5 Time Series

Visualize the temporal variation of PM2.5 concentrations.

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    df["time"],
    df["PM25"]
)

plt.xlabel("Time")
plt.ylabel("PM2.5")

plt.show()

## 6. Export & Summary

### 6.1 Export Cleaned Dataset

Export the final cleaned dataset for subsequent modeling and analysis.

In [152]:
OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final_dataset.csv"
)

OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Final dataset exported to:")
print(
    OUTPUT_FILE.relative_to(PROJECT_ROOT)
)

Final dataset exported to:
data/processed/final_dataset.csv


### 6.2 Final Summary

Summarize the final structure and quality of the cleaned dataset.

In [154]:
print("Final dataset summary")

print("\nRows:", len(df))
print("Columns:", df.shape[1])

print("\nRemaining missing values:",df.isna().sum().sum())

print("\nTime range:",df["time"].min(), "to",df["time"].max())

Final dataset summary

Rows: 220
Columns: 41

Remaining missing values: 0

Time range: 2026-03-01 03:00:00 to 2026-03-12 22:00:00
